# Demo IRISeq Pipeline: FASTQ to Fig. 2c-style Plots

This notebook is rebuilt from an empty notebook. It follows the author's IRISeq analysis logic with the local modules under `/p1/zulab_users/jtian/my_jupyter/IRI-seq/Module`.

Input FASTQ folder: `/p2/zulab/jtian/data/IRISeq/demo/Data/GSE270383/`  
Output folder: `/p2/zulab/jtian/data/IRISeq/demo/output/`

The notebook has two analysis branches that meet at the final figure:

1. cDNA FASTQ -> receiver bead x gene count matrix -> Scanpy expression UMAP and Leiden labels.
2. bead interaction FASTQ -> receiver bead x sender bead connection matrix -> spatial reconstruction UMAP.
3. Merge by receiver bead barcode -> draw the Fig. 2c-style pair of plots.

The manuscript's Fig. 2c uses cell-type labels from RCTD/single-cell reference annotation. This demo is fully Python-based, so it uses Leiden clusters as reproducible labels unless an optional RCTD weight table is supplied.


In [ ]:
from pathlib import Path
import gzip
import hashlib
import importlib
import os
import shutil
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import umap
from scipy.io import mmwrite
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=110, facecolor='white')


In [ ]:
# -------------------------
# User-facing configuration
# -------------------------
PROJECT_DIR = Path('/p1/zulab_users/jtian/my_jupyter/IRI-seq')
MODULE_SOURCE_DIR = PROJECT_DIR / 'Module'
RAW_DATA_DIR = Path('/p2/zulab/jtian/data/IRISeq/demo/Data/GSE270383')
OUTPUT_DIR = Path('/p2/zulab/jtian/data/IRISeq/demo/output')
REFERENCE_DIR = Path('/p2/zulab/jtian/data/IRISeq/reference')

# The demo GEO/SRA layout contains one cDNA pair and one bead-interaction pair.
CDNA_SAMPLE = 'GSM8341603_cDNA'
BEAD_SAMPLE = 'GSM8341604_interaction'
CDNA_SAMPLES = [CDNA_SAMPLE]
BEAD_SAMPLES = [BEAD_SAMPLE]

FASTQ_SOURCE_FILES = {
    CDNA_SAMPLE: {
        'run_accession': 'SRR29481264',
        'description': 'WT mouse hippocampus demo cDNA',
        'R1': RAW_DATA_DIR / 'SRR29481264_1.fastq.gz',
        'R2': RAW_DATA_DIR / 'SRR29481264_2.fastq.gz',
    },
    BEAD_SAMPLE: {
        'run_accession': 'SRR29481263',
        'description': 'WT mouse hippocampus demo bead interaction',
        'R1': RAW_DATA_DIR / 'SRR29481263_1.fastq.gz',
        'R2': RAW_DATA_DIR / 'SRR29481263_2.fastq.gz',
    },
}

EXPECTED_MD5 = {
    'SRR29481264_1.fastq.gz': 'db50e6caa46c1aaf07d0202512f7e9b7',
    'SRR29481264_2.fastq.gz': 'fedc4fa756a97828a4acc8ca23733f1d',
    'SRR29481263_1.fastq.gz': '2e255c3c5704b6cd6fdfe366c19910b1',
    'SRR29481263_2.fastq.gz': 'b82420ca851918e5036d114cbd0ef098',
}

PICKLE_DIR = PROJECT_DIR / 'IRIS' / 'pickle'
BARCODE_1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_1.pickle'
BARCODE_2_FILE = PICKLE_DIR / 'Spatial_R2_barcode_2.pickle'
BARCODE_3_FILE = PICKLE_DIR / 'Spatial_R2_barcode_3.pickle'
BARCODE_4_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4.pickle'
BARCODE_4_BEAD1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4_bead1.pickle'

STAR_INDEX = REFERENCE_DIR / 'STAR_mm10_GENCODE_M25'
GENE_REFERENCE_PICKLE = REFERENCE_DIR / 'Gene_annotation' / 'mm10_GENCODE_M25_Gene_reference.pickle'

# Tool paths. Change these if your Jupyter kernel does not have them on PATH.
STAR = shutil.which('STAR') or 'STAR'
SAMTOOLS = shutil.which('samtools') or 'samtools'
CUTADAPT = shutil.which('cutadapt') or 'cutadapt'

# Runtime controls. Keep FORCE flags False to reuse existing successful outputs.
RUN_CDNA_PIPELINE = True
RUN_BEAD_PIPELINE = True
FORCE_RERUN_CDNA = False
FORCE_RERUN_BEADS = False

CORE_CDNA = 2
CORE_BEADS = 10
UMI_LIMIT_FOR_ADATA = 50

# Demo QC thresholds chosen from the existing demo quality distribution.
QC_THRESHOLDS = {
    CDNA_SAMPLE: {'min_counts': 100, 'min_genes': 90, 'max_pct_mt': 8.0},
}

MIN_CONNECTION_UMIS = 7
MAX_SVD_COMPONENTS = 600
UMAP_RANDOM_STATE = 42

SAMPLE_SHEET_DIR = OUTPUT_DIR / 'sample_sheets'
SCRIPT_DIR = OUTPUT_DIR / 'Scripts'
MODULE_DIR = SCRIPT_DIR / 'EasySpatial'
FASTQ_DIR = OUTPUT_DIR / 'prepared_fastq'
REPORT_DIR = OUTPUT_DIR / 'Report'

CDNA_OUTPUT = OUTPUT_DIR / 'cDNA'
BEAD_OUTPUT = OUTPUT_DIR / 'beads'
CONNECTION_OUTPUT = OUTPUT_DIR / 'connections'
FIGURE_OUTPUT = OUTPUT_DIR / 'figures'

for directory in [OUTPUT_DIR, SAMPLE_SHEET_DIR, SCRIPT_DIR, MODULE_DIR, FASTQ_DIR, REPORT_DIR,
                  CDNA_OUTPUT, BEAD_OUTPUT, CONNECTION_OUTPUT, FIGURE_OUTPUT]:
    directory.mkdir(parents=True, exist_ok=True)

print('RAW_DATA_DIR =', RAW_DATA_DIR)
print('OUTPUT_DIR   =', OUTPUT_DIR)
print('STAR         =', STAR)
print('SAMTOOLS     =', SAMTOOLS)
print('CUTADAPT     =', CUTADAPT)


## 1. Check Raw FASTQ Files

This cell verifies file presence, file size, and MD5 values before the analysis. The old demultiplexing step is skipped because this demo already provides sample-level paired FASTQ files from GEO/SRA.


In [ ]:
raw_rows = []
for sample, info in FASTQ_SOURCE_FILES.items():
    for read in ['R1', 'R2']:
        path = info[read]
        md5 = None
        ok = False
        if path.exists():
            h = hashlib.md5()
            with open(path, 'rb') as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                    h.update(chunk)
            md5 = h.hexdigest()
            ok = md5 == EXPECTED_MD5[path.name]
        raw_rows.append({
            'sample': sample,
            'run_accession': info['run_accession'],
            'read': read,
            'path': str(path),
            'exists': path.exists(),
            'size_GB': round(path.stat().st_size / 1024**3, 3) if path.exists() else None,
            'expected_md5': EXPECTED_MD5[path.name],
            'observed_md5': md5,
            'md5_ok': ok,
        })

raw_fastq_qc = pd.DataFrame(raw_rows)
display(raw_fastq_qc)
assert raw_fastq_qc['exists'].all(), 'At least one expected FASTQ is missing.'
assert raw_fastq_qc['md5_ok'].all(), 'At least one FASTQ failed MD5 validation.'


In [ ]:
# Preview the first FASTQ record from each raw file. This is a format sanity check only.
for sample, info in FASTQ_SOURCE_FILES.items():
    for read in ['R1', 'R2']:
        path = info[read]
        print('\n#', sample, read, path)
        with gzip.open(path, 'rt') as handle:
            for _ in range(4):
                print(handle.readline().rstrip())


## 2. Prepare FASTQ Names and Sample Sheets

The local modules expect files named like `sample.R1.fastq.gz`, `sample.R2.fastq.gz`, and for bead interaction data also `sample.R3.fastq.gz`. The demo interaction data has only two FASTQ files, so `R3` is linked to the original interaction `R2`, matching the logic used in the previous demo notebook.


In [ ]:
# Recreate lightweight symlinks in the output folder. The raw files are not copied unless symlinks fail.
for sample in CDNA_SAMPLES:
    info = FASTQ_SOURCE_FILES[sample]
    for read in ['R1', 'R2']:
        src = info[read]
        dst = FASTQ_DIR / f'{sample}.{read}.fastq.gz'
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        try:
            dst.symlink_to(src)
        except OSError:
            shutil.copy2(src, dst)

for sample in BEAD_SAMPLES:
    info = FASTQ_SOURCE_FILES[sample]
    links = [
        (info['R1'], FASTQ_DIR / f'{sample}.R1.fastq.gz'),
        (info['R2'], FASTQ_DIR / f'{sample}.R2.fastq.gz'),
        (info['R2'], FASTQ_DIR / f'{sample}.R3.fastq.gz'),
    ]
    for src, dst in links:
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        try:
            dst.symlink_to(src)
        except OSError:
            shutil.copy2(src, dst)

CDNA_SAMPLE_FILE = SAMPLE_SHEET_DIR / 'cDNA_samples.txt'
BEAD_SAMPLE_FILE = SAMPLE_SHEET_DIR / 'bead_samples.txt'
CDNA_SAMPLE_FILE.write_text('\n'.join(CDNA_SAMPLES) + '\n')
BEAD_SAMPLE_FILE.write_text('\n'.join(BEAD_SAMPLES) + '\n')

prepared_rows = []
for path in sorted(FASTQ_DIR.glob('*.fastq.gz')):
    prepared_rows.append({
        'prepared_fastq': str(path),
        'target': str(path.resolve()) if path.is_symlink() else str(path),
        'size_GB': round(path.stat().st_size / 1024**3, 3),
    })
display(pd.DataFrame(prepared_rows))
print('cDNA sample sheet:', CDNA_SAMPLE_FILE)
print('bead sample sheet:', BEAD_SAMPLE_FILE)


## 3. Copy and Import Local Module Functions

The source module filenames contain spaces and parentheses, so this cell copies them to importable Python filenames under the output folder. The analysis still uses the functions from `/p1/zulab_users/jtian/my_jupyter/IRI-seq/Module`.


In [ ]:
module_map = {
    MODULE_SOURCE_DIR / 'DNAcode' / 'Spatial_UMI_barcode_extraction (1).py': MODULE_DIR / 'Spatial_UMI_barcode_extraction.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Fastq_trim_multi_files (1).py': MODULE_DIR / 'Fastq_trim_multi_files.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'STAR.py': MODULE_DIR / 'STAR.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Sam_filter_multi_files (1).py': MODULE_DIR / 'Sam_filter_multi_files.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Sam_rm_dup_barcode_UMI_multi_files (2).py': MODULE_DIR / 'Sam_rm_dup_barcode_UMI_multi_files.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Sam_gene_counting_multi_files (1).py': MODULE_DIR / 'Sam_gene_counting_multi_files.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Summary_gene_count_multi_files (1).py': MODULE_DIR / 'Summary_gene_count_multi_files.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Generate_adata (1).py': MODULE_DIR / 'Generate_adata.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'Count_reads.py': MODULE_DIR / 'Count_reads.py',
    MODULE_SOURCE_DIR / 'DNAcode' / 'File_functions (1).py': MODULE_DIR / 'File_functions.py',
    MODULE_SOURCE_DIR / 'Beadcode' / 'UMI_barcode_extraction (2).py': MODULE_DIR / 'Bead_UMI_barcode_extraction.py',
    MODULE_SOURCE_DIR / 'Beadcode' / 'spatial_barcode_extraction (3).py': MODULE_DIR / 'Bead_spatial_barcode_extraction.py',
    MODULE_SOURCE_DIR / 'Beadcode' / 'Remove_duplicate_barcode (2).py': MODULE_DIR / 'Bead_Remove_duplicate_barcode.py',
}

for src, dst in module_map.items():
    assert src.exists(), f'Missing module source: {src}'
    shutil.copy2(src, dst)

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

Spatial_UMI_barcode_extraction = importlib.import_module('Spatial_UMI_barcode_extraction')
Fastq_trim_multi_files = importlib.import_module('Fastq_trim_multi_files')
STAR_module = importlib.import_module('STAR')
Sam_filter_multi_files = importlib.import_module('Sam_filter_multi_files')
Sam_rm_dup_barcode_UMI_multi_files = importlib.import_module('Sam_rm_dup_barcode_UMI_multi_files')
Sam_gene_counting_multi_files = importlib.import_module('Sam_gene_counting_multi_files')
Summary_gene_count_multi_files = importlib.import_module('Summary_gene_count_multi_files')
Generate_adata = importlib.import_module('Generate_adata')
Count_reads = importlib.import_module('Count_reads')
Bead_UMI_barcode_extraction = importlib.import_module('Bead_UMI_barcode_extraction')
Bead_spatial_barcode_extraction = importlib.import_module('Bead_spatial_barcode_extraction')
Bead_Remove_duplicate_barcode = importlib.import_module('Bead_Remove_duplicate_barcode')

# The bead spatial module references this global name internally, so define it explicitly after import.
Bead_spatial_barcode_extraction.list_barcode_4_file2 = str(BARCODE_4_FILE)

print('Prepared and imported modules in:', MODULE_DIR)


## 4. cDNA Pipeline: FASTQ to Gene Count Matrix

This is the transcriptome branch. The cDNA R1 barcode/UMI is attached to R2, R2 is trimmed and aligned to the mouse genome, duplicates are removed, and UMI counts are summarized per receiver bead and gene.


In [ ]:
FASTQ_BARCODE_ATTACHED = CDNA_OUTPUT / 'Fastq_barcode_attached'
FASTQ_TRIMMED = CDNA_OUTPUT / 'Fastq_trimmed'
SAM_STAR = CDNA_OUTPUT / 'Sam_STAR'
SAM_FILTERED = CDNA_OUTPUT / 'Sam_filtered'
SAM_RMDUP = CDNA_OUTPUT / 'Sam_rmdup'
BED_GENE_COUNT = CDNA_OUTPUT / 'Bed_gene_count'
SUMMARY_GENE_COUNT = CDNA_OUTPUT / 'Summary_gene_count'
ADATA_FOLDER = CDNA_OUTPUT / 'Adata'

for directory in [FASTQ_BARCODE_ATTACHED, FASTQ_TRIMMED, SAM_STAR, SAM_FILTERED,
                  SAM_RMDUP, BED_GENE_COUNT, SUMMARY_GENE_COUNT, ADATA_FOLDER]:
    directory.mkdir(parents=True, exist_ok=True)

adata_full_path = ADATA_FOLDER / 'adata_full.h5ad'

if FORCE_RERUN_CDNA:
    for directory in [FASTQ_BARCODE_ATTACHED, FASTQ_TRIMMED, SAM_STAR, SAM_FILTERED,
                      SAM_RMDUP, BED_GENE_COUNT, SUMMARY_GENE_COUNT, ADATA_FOLDER]:
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

if RUN_CDNA_PIPELINE and (FORCE_RERUN_CDNA or not adata_full_path.exists()):
    assert STAR_INDEX.exists(), f'Missing STAR index: {STAR_INDEX}'
    assert GENE_REFERENCE_PICKLE.exists(), f'Missing gene reference pickle: {GENE_REFERENCE_PICKLE}'

    Spatial_UMI_barcode_extraction.extract_spatial_barcode_files(
        str(FASTQ_DIR), str(CDNA_SAMPLE_FILE), str(FASTQ_BARCODE_ATTACHED), CORE_CDNA,
        str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
    )

    Fastq_trim_multi_files.Fastq_trim_files(
        str(FASTQ_BARCODE_ATTACHED), str(CDNA_SAMPLE_FILE), str(FASTQ_TRIMMED), CORE_CDNA,
        adapter_seq='AAAAAAAA', cutadapt_path=str(CUTADAPT),
    )

    STAR_module.Fastq_star_alignment_multi_files(
        str(FASTQ_TRIMMED), str(CDNA_SAMPLE_FILE), str(SAM_STAR), CORE_CDNA,
        index=str(STAR_INDEX), star_path=str(STAR),
    )

    Sam_filter_multi_files.Sam_filter_files(
        str(SAM_STAR), str(CDNA_SAMPLE_FILE), str(SAM_FILTERED), CORE_CDNA,
        samtools_path=str(SAMTOOLS),
    )

    Sam_rm_dup_barcode_UMI_multi_files.rm_dup_files(
        str(SAM_FILTERED), str(CDNA_SAMPLE_FILE), str(SAM_RMDUP), CORE_CDNA,
    )

    Sam_gene_counting_multi_files.scRNA_count_parallel(
        str(SAM_RMDUP), str(CDNA_SAMPLE_FILE), str(BED_GENE_COUNT), str(GENE_REFERENCE_PICKLE), CORE_CDNA,
    )

    Summary_gene_count_multi_files.Gene_count_summary(
        str(BED_GENE_COUNT), str(CDNA_SAMPLE_FILE), str(SUMMARY_GENE_COUNT),
    )

    Generate_adata.generate_adata_from_gene_count(
        str(SUMMARY_GENE_COUNT), str(BED_GENE_COUNT / 'gene_anno.csv'), str(ADATA_FOLDER), UMI_LIMIT_FOR_ADATA,
    )
else:
    print('Skipping cDNA pipeline because existing AnnData was found:', adata_full_path)


In [ ]:
# Summarize read counts after the cDNA pipeline.
df_count = pd.DataFrame({'Sample_name': CDNA_SAMPLES})
df_count['Count_Fastq_raw'] = Count_reads.Fastq_count_reads_files(str(FASTQ_DIR), str(CDNA_SAMPLE_FILE))
df_tmp = Count_reads.Count_Align_STAR_files(str(SAM_STAR), str(CDNA_SAMPLE_FILE))
df_count['Count_Fastq_filtered'] = Count_reads.Fastq_count_reads_files(str(FASTQ_BARCODE_ATTACHED), str(CDNA_SAMPLE_FILE))
df_count['Count_Fastq_trimmed'] = list(df_tmp[0])
df_count['Count_Sam_mapped'] = list(df_tmp[1] + df_tmp[2])
df_count['Count_Sam_unique'] = list(df_tmp[1])
df_count['Count_Sam_rmdup'] = Count_reads.SAM_count_reads_files(str(SAM_RMDUP), str(CDNA_SAMPLE_FILE))
df_count['Count_Sam_annotated'] = Count_reads.count_mapped_reads_files(str(BED_GENE_COUNT), str(CDNA_SAMPLE_FILE))
df_count['Count_Cell_reads'] = Count_reads.Count_cell_reads(str(ADATA_FOLDER))

df_reads_path = ADATA_FOLDER / 'df_reads.csv'
df_count.to_csv(df_reads_path, index=False)
display(df_count)
print('Saved', df_reads_path)


## 5. Expression UMAP and Cluster Labels

This cell performs the expression-side analysis analogous to the left panel of Fig. 2c. Each point is a receiver bead. Similar expression profiles are placed close together by UMAP.


In [ ]:
adata = sc.read_h5ad(ADATA_FOLDER / 'adata_full.h5ad')
print(adata)

# Extract receiver barcode from the cDNA cell name. Example: sample.A-B-C-D -> ABCD concatenated barcode.
adata.obs['sample_source'] = adata.obs_names.to_series().str.split('.').str[0].values
adata.obs['receiver_barcode'] = adata.obs_names.to_series().str.split('.').str[-1].str.replace('-', '', regex=False).values

# Keep mouse genes and switch from Ensembl IDs to gene symbols for easier marker plotting.
if 'Gene_name' in adata.var.columns:
    mouse_mask = adata.var_names.astype(str).str.startswith('ENSMUSG')
    adata = adata[:, mouse_mask].copy()
    adata.var_names = adata.var['Gene_name'].astype(str).values
    adata.var_names_make_unique()

adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Apply sample-specific QC thresholds.
keep = np.ones(adata.n_obs, dtype=bool)
for sample in CDNA_SAMPLES:
    threshold = QC_THRESHOLDS[sample]
    sample_mask = adata.obs['sample_source'].eq(sample).values
    sample_keep = (
        (adata.obs['total_counts'].values >= threshold['min_counts']) &
        (adata.obs['n_genes_by_counts'].values >= threshold['min_genes']) &
        (adata.obs['pct_counts_mt'].fillna(0).values <= threshold['max_pct_mt'])
    )
    keep[sample_mask] = sample_keep[sample_mask]

qc_summary_rows = []
for sample in CDNA_SAMPLES:
    before = int((adata.obs['sample_source'] == sample).sum())
    after = int(((adata.obs['sample_source'] == sample).values & keep).sum())
    threshold = QC_THRESHOLDS[sample]
    qc_summary_rows.append({
        'sample': sample,
        'n_barcodes_before': before,
        'n_barcodes_after': after,
        'n_barcodes_removed': before - after,
        'barcode_retention_pct': after / before * 100 if before else 0,
        'min_counts': threshold['min_counts'],
        'min_genes': threshold['min_genes'],
        'max_pct_mt': threshold['max_pct_mt'],
    })

qc_summary = pd.DataFrame(qc_summary_rows)
qc_summary_path = ADATA_FOLDER / 'cDNA_qc_filtering_summary_by_sample.csv'
qc_summary.to_csv(qc_summary_path, index=False)
display(qc_summary)

adata_qc = adata[keep].copy()
adata_qc.layers['counts'] = adata_qc.X.copy()

sc.pp.filter_genes(adata_qc, min_cells=3)
sc.pp.normalize_total(adata_qc, target_sum=1e4)
sc.pp.log1p(adata_qc)
adata_qc.raw = adata_qc
sc.pp.highly_variable_genes(adata_qc, n_top_genes=min(3000, adata_qc.n_vars), flavor='seurat')
adata_qc = adata_qc[:, adata_qc.var['highly_variable']].copy()
sc.pp.scale(adata_qc, max_value=10)

n_pcs = min(50, adata_qc.n_obs - 1, adata_qc.n_vars - 1)
sc.tl.pca(adata_qc, n_comps=n_pcs, svd_solver='arpack')
sc.pp.neighbors(adata_qc, n_neighbors=20, n_pcs=min(30, n_pcs), random_state=UMAP_RANDOM_STATE)
sc.tl.umap(adata_qc, min_dist=0.1, random_state=UMAP_RANDOM_STATE)
sc.tl.leiden(adata_qc, resolution=1.0, key_added='leiden')

# Optional Python-compatible label hook. If no RCTD CSV exists, use Leiden clusters.
rctd_weights_csv = OUTPUT_DIR / 'annotations' / 'rctd_weights.csv'
if rctd_weights_csv.exists():
    weights = pd.read_csv(rctd_weights_csv, index_col=0)
    common = adata_qc.obs_names.intersection(weights.index)
    adata_qc.obs['fig2c_label'] = adata_qc.obs['leiden'].astype(str)
    adata_qc.obs.loc[common, 'fig2c_label'] = weights.loc[common].idxmax(axis=1).astype(str)
    print('Using RCTD labels from', rctd_weights_csv)
else:
    adata_qc.obs['fig2c_label'] = 'Leiden_' + adata_qc.obs['leiden'].astype(str)
    print('No Python-readable RCTD table found; using Leiden clusters as Fig. 2c labels.')

qc_processed_path = ADATA_FOLDER / 'adata_qc_processed_umap_leiden.h5ad'
adata_qc.write_h5ad(qc_processed_path)
print('Saved', qc_processed_path)
print(adata_qc)


In [ ]:
# Plot the expression UMAP, equivalent to the left coordinate system in Fig. 2c.
adata_qc = sc.read_h5ad(ADATA_FOLDER / 'adata_qc_processed_umap_leiden.h5ad')
fig_dir = CDNA_OUTPUT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

sc.pl.umap(adata_qc, color='fig2c_label', frameon=False, legend_loc='right margin', show=False)
fig = plt.gcf()
expression_umap_path = fig_dir / 'fig2c_style_expression_umap_by_label.png'
fig.savefig(expression_umap_path, dpi=220, bbox_inches='tight')
plt.show()

expr_coords = pd.DataFrame(
    adata_qc.obsm['X_umap'],
    index=adata_qc.obs_names,
    columns=['expression_UMAP1', 'expression_UMAP2'],
)
expr_meta = adata_qc.obs[['receiver_barcode', 'fig2c_label', 'leiden', 'total_counts', 'n_genes_by_counts', 'pct_counts_mt']].join(expr_coords)
expr_meta_path = ADATA_FOLDER / 'fig2c_expression_umap_metadata.csv'
expr_meta.to_csv(expr_meta_path)
print('Saved', expression_umap_path)
print('Saved', expr_meta_path)


## 6. Bead Interaction Pipeline: FASTQ to Spatial Reconstruction Coordinates

This is the spatial branch. It extracts receiver-sender bead connections, removes duplicate molecular connections, builds the receiver bead x sender bead matrix, and uses PCA/SVD plus UMAP to infer a two-dimensional spatial reconstruction.


In [ ]:
UMI_ATTACH = BEAD_OUTPUT / 'UMI_attach'
SPATIAL_BARCODE = BEAD_OUTPUT / 'Spatial_barcode_extraction'
DEDUPLICATE_SPATIAL = BEAD_OUTPUT / 'Spatial_barcode_rmdup'
BEAD_REPORT_FOLDER = BEAD_OUTPUT / 'report' / 'read_num_spatial_barcode'

for directory in [UMI_ATTACH, SPATIAL_BARCODE, DEDUPLICATE_SPATIAL, BEAD_REPORT_FOLDER]:
    directory.mkdir(parents=True, exist_ok=True)

spatial_rmdup_path = DEDUPLICATE_SPATIAL / f'{BEAD_SAMPLE}.spatial.csv.gz'

if FORCE_RERUN_BEADS:
    for directory in [UMI_ATTACH, SPATIAL_BARCODE, DEDUPLICATE_SPATIAL, BEAD_REPORT_FOLDER]:
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)

if RUN_BEAD_PIPELINE and (FORCE_RERUN_BEADS or not spatial_rmdup_path.exists()):
    Bead_UMI_barcode_extraction.extract_spatial_barcode_files(
        str(FASTQ_DIR), str(BEAD_SAMPLE_FILE), str(UMI_ATTACH), CORE_BEADS,
        str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
    )
    Bead_spatial_barcode_extraction.list_barcode_4_file2 = str(BARCODE_4_FILE)
    Bead_spatial_barcode_extraction.extract_spatial_barcode_files(
        str(UMI_ATTACH), str(BEAD_SAMPLE_FILE), str(SPATIAL_BARCODE), CORE_BEADS,
        str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_FILE),
    )
    Bead_Remove_duplicate_barcode.remove_duplicates_files(
        str(SPATIAL_BARCODE), str(BEAD_SAMPLE_FILE), str(DEDUPLICATE_SPATIAL), CORE_BEADS,
    )
else:
    print('Skipping bead pipeline because existing deduplicated spatial file was found:', spatial_rmdup_path)


In [ ]:
# Count reads/molecular connections after each bead-interaction step.
bead_read_rows = []
for sample in BEAD_SAMPLES:
    raw_lines = sum(1 for _ in gzip.open(FASTQ_DIR / f'{sample}.R2.fastq.gz', 'rt'))
    umi_lines = sum(1 for _ in gzip.open(UMI_ATTACH / f'{sample}.R2.fastq.gz', 'rt'))
    spatial_lines = sum(1 for _ in gzip.open(SPATIAL_BARCODE / f'{sample}.spatial.txt.gz', 'rt'))
    dedup_lines = sum(1 for _ in gzip.open(DEDUPLICATE_SPATIAL / f'{sample}.spatial.csv.gz', 'rt'))
    bead_read_rows.append({
        'sample': sample,
        'total_reads': raw_lines // 4,
        'Filtering_bead1_barcode': umi_lines // 5,
        'Filtering_bead2_barcode': spatial_lines,
        'Remove_duplicates': dedup_lines,
    })

bead_read_number = pd.DataFrame(bead_read_rows)
bead_read_number_path = BEAD_REPORT_FOLDER / 'read_number.csv'
bead_read_number.to_csv(bead_read_number_path, index=False)
display(bead_read_number)
print('Saved', bead_read_number_path)


In [ ]:
# Build a sparse receiver bead x sender bead connection matrix.
def read_spatial_connections(path):
    raw = pd.read_csv(path, compression='gzip', header=None, dtype=str)
    if raw.shape[1] == 1:
        parsed = raw.iloc[:, 0].str.split(',', n=2, expand=True)
    else:
        parsed = raw.iloc[:, :3].copy()
    parsed.columns = ['Bead1_seq', 'UMI', 'Bead2_seq']
    parsed = parsed.dropna()
    for column in parsed.columns:
        parsed[column] = parsed[column].astype(str).str.strip()
    parsed = parsed[(parsed['Bead1_seq'] != '') & (parsed['Bead2_seq'] != '') & (parsed['UMI'] != '')]
    return parsed

connection_rows = []
for sample in BEAD_SAMPLES:
    path = DEDUPLICATE_SPATIAL / f'{sample}.spatial.csv.gz'
    df = read_spatial_connections(path)
    grouped = (
        df.groupby(['Bead1_seq', 'Bead2_seq'], observed=True)['UMI']
        .nunique()
        .reset_index(name='n_umi')
    )
    grouped = grouped[grouped['n_umi'] >= MIN_CONNECTION_UMIS].copy()
    grouped['sample'] = sample
    connection_rows.append(grouped)
    print(sample, 'raw connections=', len(df), 'filtered receiver-sender edges=', grouped.shape[0])

connection_edges = pd.concat(connection_rows, ignore_index=True)
edge_path = CONNECTION_OUTPUT / 'receiver_sender_edges_min_umi.csv.gz'
connection_edges.to_csv(edge_path, index=False, compression='gzip')

receiver_index = pd.Index(sorted(connection_edges['Bead1_seq'].unique()), name='receiver_barcode')
sender_index = pd.Index(sorted(connection_edges['Bead2_seq'].unique()), name='sender_barcode')
receiver_codes = receiver_index.get_indexer(connection_edges['Bead1_seq'])
sender_codes = sender_index.get_indexer(connection_edges['Bead2_seq'])
connection_matrix = sp.csr_matrix(
    (connection_edges['n_umi'].astype(np.float32), (receiver_codes, sender_codes)),
    shape=(len(receiver_index), len(sender_index)),
)

sp.save_npz(CONNECTION_OUTPUT / 'receiver_sender_connection_matrix_min_umi.npz', connection_matrix)
receiver_index.to_frame(index=False).to_csv(CONNECTION_OUTPUT / 'receiver_barcodes.csv', index=False)
sender_index.to_frame(index=False).to_csv(CONNECTION_OUTPUT / 'sender_barcodes.csv', index=False)

print('connection_matrix shape =', connection_matrix.shape)
print('nonzero edges =', connection_matrix.nnz)
print('Saved', edge_path)


In [ ]:
# Reconstruct receiver bead positions from the connection matrix.
connection_matrix = sp.load_npz(CONNECTION_OUTPUT / 'receiver_sender_connection_matrix_min_umi.npz')
receiver_index = pd.read_csv(CONNECTION_OUTPUT / 'receiver_barcodes.csv')['receiver_barcode'].astype(str)

matrix_log = connection_matrix.copy().astype(np.float32)
matrix_log.data = np.log1p(matrix_log.data)
scaler = StandardScaler(with_mean=False)
matrix_scaled = scaler.fit_transform(matrix_log)

max_valid_components = min(matrix_scaled.shape[0] - 1, matrix_scaled.shape[1] - 1)
n_components = min(MAX_SVD_COMPONENTS, max_valid_components)
assert n_components >= 2, f'Connection matrix is too small for reconstruction: {matrix_scaled.shape}'

svd = TruncatedSVD(n_components=n_components, random_state=UMAP_RANDOM_STATE)
connection_pca = svd.fit_transform(matrix_scaled)
explained = pd.DataFrame({
    'component': np.arange(1, n_components + 1),
    'explained_variance_ratio': svd.explained_variance_ratio_,
    'cumulative_explained_variance': np.cumsum(svd.explained_variance_ratio_),
})
explained_path = CONNECTION_OUTPUT / 'connection_svd_explained_variance.csv'
explained.to_csv(explained_path, index=False)

graph_neighbors = min(20, max(2, connection_pca.shape[0] - 1))
spatial_mapper = umap.UMAP(
    n_neighbors=graph_neighbors,
    min_dist=0.2,
    spread=0.6,
    metric='euclidean',
    random_state=UMAP_RANDOM_STATE,
    n_epochs=500,
    verbose=True,
)
spatial_coords = spatial_mapper.fit_transform(connection_pca)

spatial_umap = pd.DataFrame({
    'connection_barcode': receiver_index.values,
    'spatial_UMAP1': spatial_coords[:, 0],
    'spatial_UMAP2': spatial_coords[:, 1],
})
spatial_umap_path = CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv'
spatial_umap.to_csv(spatial_umap_path, index=False)

print('Saved', explained_path)
print('Saved', spatial_umap_path)
display(spatial_umap.head())


In [ ]:
# Plot the spatial reconstruction by connection density.
spatial_umap = pd.read_csv(CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv')
edge_counts = pd.read_csv(CONNECTION_OUTPUT / 'receiver_sender_edges_min_umi.csv.gz')
receiver_degree = edge_counts.groupby('Bead1_seq').size().rename('n_sender_beads').reset_index()
spatial_plot_df = spatial_umap.merge(receiver_degree, left_on='connection_barcode', right_on='Bead1_seq', how='left')
spatial_plot_df['n_sender_beads'] = spatial_plot_df['n_sender_beads'].fillna(0)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    spatial_plot_df['spatial_UMAP1'], spatial_plot_df['spatial_UMAP2'],
    c=spatial_plot_df['n_sender_beads'], s=2, cmap='viridis', linewidths=0,
)
ax.set_aspect('equal', adjustable='datalim')
ax.set_title('Spatial reconstruction from bead-bead interactions')
ax.set_xlabel('Spatial UMAP1')
ax.set_ylabel('Spatial UMAP2')
fig.colorbar(scatter, ax=ax, label='Connected sender beads')
fig.tight_layout()
spatial_plot_path = CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap_by_connection_density.png'
fig.savefig(spatial_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved', spatial_plot_path)


## 7. Fig. 2c-style Merge and Final Plots

The expression UMAP and spatial UMAP are merged by receiver bead barcode. The left panel uses expression-derived coordinates. The right panel uses bead-interaction-derived spatial reconstruction coordinates. Both panels use the same labels and colors.


In [ ]:
adata_qc = sc.read_h5ad(ADATA_FOLDER / 'adata_qc_processed_umap_leiden.h5ad')
spatial_umap = pd.read_csv(CONNECTION_OUTPUT / f'{BEAD_SAMPLE}_spatial_reconstruction_umap.csv')

expr_df = adata_qc.obs[['receiver_barcode', 'fig2c_label', 'leiden', 'total_counts', 'n_genes_by_counts']].copy()
expr_df['cell_name'] = expr_df.index
expr_df['expression_UMAP1'] = adata_qc.obsm['X_umap'][:, 0]
expr_df['expression_UMAP2'] = adata_qc.obsm['X_umap'][:, 1]

fig2c_df = expr_df.merge(
    spatial_umap,
    left_on='receiver_barcode',
    right_on='connection_barcode',
    how='inner',
)

fig2c_df_path = FIGURE_OUTPUT / 'fig2c_style_matched_expression_and_spatial_coordinates.csv'
fig2c_df.to_csv(fig2c_df_path, index=False)

print('QC-filtered expression beads:', expr_df.shape[0])
print('Spatially reconstructed receiver beads:', spatial_umap.shape[0])
print('Matched beads for Fig. 2c-style plot:', fig2c_df.shape[0])
assert fig2c_df.shape[0] > 0, 'No matching receiver barcodes between cDNA and bead-interaction outputs.'

display(fig2c_df.head())
print('Saved', fig2c_df_path)


In [ ]:
# Draw the paired Fig. 2c-style result.
labels = sorted(fig2c_df['fig2c_label'].astype(str).unique())
palette = sc.pl.palettes.default_102
color_map = {label: palette[i % len(palette)] for i, label in enumerate(labels)}
colors = fig2c_df['fig2c_label'].astype(str).map(color_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

axes[0].scatter(
    fig2c_df['expression_UMAP1'], fig2c_df['expression_UMAP2'],
    c=colors, s=4, linewidths=0, alpha=0.85,
)
axes[0].set_title('Gene expression UMAP')
axes[0].set_xlabel('UMAP1 (gene expression)')
axes[0].set_ylabel('UMAP2 (gene expression)')
axes[0].set_aspect('equal', adjustable='datalim')

axes[1].scatter(
    fig2c_df['spatial_UMAP1'], fig2c_df['spatial_UMAP2'],
    c=colors, s=4, linewidths=0, alpha=0.85,
)
axes[1].set_title('Spatial reconstruction')
axes[1].set_xlabel('UMAP1 (bead-bead interaction)')
axes[1].set_ylabel('UMAP2 (bead-bead interaction)')
axes[1].set_aspect('equal', adjustable='datalim')

handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[label], markersize=6, label=label)
    for label in labels
]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.01, 0.5), title='Label')

fig2c_plot_path = FIGURE_OUTPUT / 'fig2c_style_expression_umap_and_spatial_reconstruction.png'
fig.savefig(fig2c_plot_path, dpi=260, bbox_inches='tight')
plt.show()
print('Saved', fig2c_plot_path)


## 8. Export Matrix Files for R/Seurat Compatibility

The author's Fig. 2 plotting notebook reads `genecount.mtx`, `df_cell.csv`, and `df_gene.csv`. This cell exports Python-derived files in the same spirit so the result can be inspected in R/Seurat if needed.


In [ ]:
seurat_export_dir = OUTPUT_DIR / 'seurat_compatible_export'
seurat_export_dir.mkdir(parents=True, exist_ok=True)

adata_export = sc.read_h5ad(ADATA_FOLDER / 'adata_qc_processed_umap_leiden.h5ad')
counts = adata_export.layers['counts'] if 'counts' in adata_export.layers else adata_export.X
if not sp.issparse(counts):
    counts = sp.csr_matrix(counts)

mmwrite(seurat_export_dir / 'genecount_qc_filtered.mtx', counts.T.tocoo())

df_cell = adata_export.obs.copy()
df_cell['cell_name_temp'] = df_cell.index
df_cell['expression_UMAP1'] = adata_export.obsm['X_umap'][:, 0]
df_cell['expression_UMAP2'] = adata_export.obsm['X_umap'][:, 1]
df_cell.to_csv(seurat_export_dir / 'df_cell_qc_filtered.csv', index=False)

pd.DataFrame({'Gene_name': adata_export.var_names}).to_csv(seurat_export_dir / 'df_gene_qc_filtered.csv', index=False)

print('Saved Seurat-compatible exports under:', seurat_export_dir)


## 9. Output Checklist

Main outputs created by this notebook:

- cDNA read-count summary: `cDNA/Adata/df_reads.csv`
- QC-filtered expression object: `cDNA/Adata/adata_qc_processed_umap_leiden.h5ad`
- bead read-count summary: `beads/report/read_num_spatial_barcode/read_number.csv`
- sparse connection matrix: `connections/receiver_sender_connection_matrix_min_umi.npz`
- spatial reconstruction table: `connections/GSM8341604_interaction_spatial_reconstruction_umap.csv`
- final paired plot: `figures/fig2c_style_expression_umap_and_spatial_reconstruction.png`

If you later provide an RCTD weight table at `output/annotations/rctd_weights.csv`, rerun the expression-label and final-plot cells to color the plots by cell-type labels instead of Leiden clusters.
